# 🎯 Fase 4: Modelo Mejorado con Insights del Paper

**Paper:** "Rise of the Machines? Intraday High-Frequency Trading Patterns"
**Autores:** Petukhina, Reule, Härdle (Humboldt-Universität Berlin)

## Features Implementadas Basadas en el Paper:
1. ✅ RSI + SMA (ya existentes)
2. ⭐ `hour_of_day` - Time-Of-Day Effects del paper
3. ⭐ `day_of_week` - Patrones semanales
4. ⭐ `is_peak_hour` - Horas pico (Asia/Europa/América)
5. ⭐ `ATR_14` - Volatility Clustering (volatilidad autocorrelacionada)

## Objetivo:
Mejorar accuracy del modelo >60% usando patrones temporales

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
import asyncpg
import json
import onnx
from onnxmltools.convert import convert_sklearn
import time
from datetime import datetime

print("✅ Librerías cargadas")

✅ Librerías cargadas


## 📡 Conexión a TimescaleDB en Raspberry Pi

**IP:** 192.168.1.94  
**Puerto:** 5432  
**Usuario:** solana_user  
**Base de datos:** solana_db

In [2]:
import os
async def load_trades(days=30):
    """Carga trades desde TimescaleDB en la Pi"""
    conn = await asyncpg.connect(
        host='192.168.1.94',  # IP fija de Raspberry Pi
        port=5432,
        user='solana_user',
        password=os.environ['DB_PASSWORD']  # defínela antes: set DB_PASSWORD=...,
        database='solana_trading'
    )
    
    query = """
        SELECT timestamp, price, size, side, trade_id 
        FROM solana_trades 
        WHERE timestamp >= NOW() - INTERVAL '{} days'
        ORDER BY timestamp ASC
    """.format(days)
    
    rows = await conn.fetch(query)
    await conn.close()
    
    df = pd.DataFrame(rows, columns=['timestamp', 'price', 'size', 'side', 'trade_id'])
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    print(f"✅ Descargados {len(df)} trades de los últimos {days} días")
    return df

## 🔧 Creación de Features Mejoradas (Paper-Based)

In [3]:
def create_enhanced_features_paper(df):
    """
    Crea features basadas en el paper "Rise of the Machines?"
    
    Features del paper implementadas:
    - RSI (momentum)
    - SMA_20, SMA_50 (tendencia)
    - Volume_MA_20 (volumen promedio)
    - hour_of_day ⭐ Time-Of-Day Effects
    - day_of_week ⭐ Patrones semanales
    - is_peak_hour ⭐ Horas pico globales
    - ATR_14 ⭐ Volatility Clustering
    """
    
    df = df.copy()
    
    # 1. RSI (Índice de Fuerza Relativa)
    def calculate_rsi(prices, period=14):
        delta = prices.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / (loss + 1e-10)
        return 100 - (100 / (1 + rs))
    
    df['RSI'] = calculate_rsi(df['price'])
    
    # 2. Medias Móviles (Tendencia)
    df['SMA_20'] = df['price'].rolling(window=20).mean()
    df['SMA_50'] = df['price'].rolling(window=50).mean()
    
    # 3. Volumen Promedio Móvil (columna 'size' de la DB)
    if 'size' in df.columns:
        df['Volume_MA_20'] = df['size'].rolling(window=20).mean()
    elif 'volume' in df.columns:
        df['Volume_MA_20'] = df['volume'].rolling(window=20).mean()
    
    # 4. ⭐ TIME-OF-DAY EFFECTS (Paper Feature)
    df['hour_of_day'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek  # 0=Monday, 6=Sunday
    
    # Definir horas pico por zona horaria (UTC)
    def is_peak_hour(hour):
        if 7 <= hour <= 16:  # Europa
            return 1
        elif 13 <= hour <= 22:  # América
            return 1
        elif 0 <= hour <= 9 or 23 == hour:  # Asia
            return 1
        else:
            return 0
    
    df['is_peak_hour'] = df['hour_of_day'].apply(is_peak_hour)
    
    # 5. ⭐ VOLATILITY CLUSTERING (Paper Feature) - ATR
    high_low = df['price'].diff().abs()
    df['ATR_14'] = high_low.rolling(window=14).mean()
    
    # 6. TARGET: ¿Subirá en próximos 5 minutos? (60 trades ~ 5 min)
    df['target'] = (df['price'].shift(-60) > df['price']).astype(int)
    
    # Eliminar filas NaN
    df = df.dropna()
    
    return df

In [4]:
# Procesar datos
print("🔧 Procesando datos con features del paper...")
trades = await load_trades(days=30)
df_features = create_enhanced_features_paper(trades)

print(f"✅ Datos procesados: {len(df_features)} filas")
print(f"\nColumnas disponibles:")
print(df_features.columns.tolist())
print(f"\nPrimeras 5 filas:")
print(df_features.head())

🔧 Procesando datos con features del paper...
✅ Descargados 5530 trades de los últimos 30 días
✅ Datos procesados: 5481 filas

Columnas disponibles:
['timestamp', 'price', 'size', 'side', 'trade_id', 'RSI', 'SMA_20', 'SMA_50', 'Volume_MA_20', 'hour_of_day', 'day_of_week', 'is_peak_hour', 'ATR_14', 'target']

Primeras 5 filas:
                          timestamp         price         size side  \
49 2026-09-30 03:23:54.097182+00:00  119.34000000  31.41000000    A   
50 2026-09-30 03:23:54.139446+00:00  119.34000000   1.89000000    A   
51 2026-09-30 03:23:54.181874+00:00  119.34000000   2.24000000    A   
52 2026-09-30 03:23:54.224341+00:00  119.34000000  41.42000000    A   
53 2026-09-30 03:23:54.267225+00:00  119.34000000   0.84000000    A   

            trade_id        RSI    SMA_20    SMA_50  Volume_MA_20  \
49   528003628169058  99.999993  119.3330  119.3584       35.0820   
50   220333877587890  99.999993  119.3305  119.3584       35.1720   
51   575019858606442  99.999993  119.33

In [5]:
# Separar features y target
feature_cols = ['RSI', 'SMA_20', 'SMA_50', 'Volume_MA_20', 
                'hour_of_day', 'day_of_week', 'is_peak_hour', 'ATR_14']

X = df_features[feature_cols].values
y = df_features['target'].values

# Escalar features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 Dataset:")
print(f"   Train: {len(X_train)} muestras")
print(f"   Test:  {len(X_test)} muestras")
print(f"   Features: {len(feature_cols)}")


📊 Dataset:
   Train: 4384 muestras
   Test:  1097 muestras
   Features: 8


In [6]:
# Entrenar Random Forest
print("\n🎯 Entrenando Random Forest con features del paper...")

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Evaluación
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
cv_scores = cross_val_score(model, X_scaled, y, cv=5)

print(f"\n📈 Resultados:")
print(f"   Accuracy Test: {accuracy:.4f}")
print(f"   CV Mean ± Std: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred))


🎯 Entrenando Random Forest con features del paper...

📈 Resultados:
   Accuracy Test: 0.9107
   CV Mean ± Std: 0.4236 ± 0.0921

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.90      0.92       647
           1       0.87      0.92      0.89       450

    accuracy                           0.91      1097
   macro avg       0.91      0.91      0.91      1097
weighted avg       0.91      0.91      0.91      1097



In [7]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
print("\n🔲 Matriz de Confusión:")
print(cm)

# Visualización opcional
try:
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Matriz de Confusión - Modelo Paper-Based')
    plt.colorbar()
    
    tick_marks = np.arange(len(np.unique(y_test)))
    plt.xticks(tick_marks, np.unique(y_test), rotation=45)
    plt.yticks(tick_marks, np.unique(y_test))
    
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                    ha="center", va="center",
                    color="white" if cm[i,j] > thresh else "black")
    
    plt.ylabel('Etiqueta Real')
    plt.xlabel('Predicción')
    plt.tight_layout()
    plt.savefig('confusion_matrix_paper.png', dpi=300)
    print("✅ Matriz guardada como confusion_matrix_paper.png")
except Exception as e:
    print(f"⚠️ Visualización no disponible: {e}")


🔲 Matriz de Confusión:
[[583  64]
 [ 34 416]]
⚠️ Visualización no disponible: No module named 'matplotlib'


In [8]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# Exportar modelo a ONNX
print("\n💾 Exportando modelo a ONNX...")

# Definir la forma de los datos de entrada: [Filas dinámicas, Número de características]
initial_type = [('float_input', FloatTensorType([None, len(feature_cols)]))]

# Convertir el modelo de Scikit-Learn a ONNX
onnx_model = convert_sklearn(model, initial_types=initial_type)

# Guardar el modelo en disco
onnx_model_path = '../models/solana_model_v1.onnx'
with open(onnx_model_path, 'wb') as f:
    f.write(onnx_model.SerializeToString())

print(f"✅ Modelo exportado con éxito a: {onnx_model_path}")


💾 Exportando modelo a ONNX...
✅ Modelo exportado con éxito a: ../models/solana_model_v1.onnx


In [9]:
# Guardar metadata
metadata = {
    'model_name': 'solana_model_v2_paper_enhanced',
    'created_at': datetime.now().isoformat(),
    'features': feature_cols,
    'accuracy': float(accuracy),
    'cv_score': float(cv_scores.mean()),
    'threshold': 0.60
}

with open('../models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ Metadata guardada en ../models/model_metadata.json")

# Comando SCP
scp_command = """
scp ../models/solana_model_v2_paper_enhanced.onnx fumi7@192.168.1.94:/home/fumi7/1TRADING/models/
"""

✅ Metadata guardada en ../models/model_metadata.json


In [10]:
!scp ../models/solana_model_v1.onnx fumi7@192.168.1.94:/home/fumi7/1TRADING/models/solana_model_v1.onnx